# Food Price Watch 02: data processing
Builds the one data table for the 8 items from `MyDrive/FoodPriceWatch_v2/raw/`: TCB prices plus calendar, weather and exchange-rate columns, one row per item and report day.
Checks it and compares it with the earlier run's panel and features.
Writes `processed/data_8.csv`, `processed/checks.json` and `reports/tables/D1_panel_summary.csv` (rates only, no prices).
Needs notebook 01 to have run. Run top to bottom and paste the last cell's output back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes fresh from GitHub main each session; Drive holds data and outputs only.
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
import sys, os, json
os.chdir('/content/foodprice_watch')
sys.path.insert(0, '/content/foodprice_watch/src')
CODE_COMMIT = !git rev-parse --short HEAD
CODE_COMMIT = CODE_COMMIT[0]
from foodwatch.paths import SETTINGS, DATA_ROOT, RAW, data, raw, sha256, write_csv
# stop if Drive is not really mounted (a failed mount leaves an empty /content/drive on the Colab machine)
assert os.path.ismount('/content/drive'), 'Google Drive is not mounted: Runtime > Disconnect and delete runtime, then rerun cell 1'
assert DATA_ROOT.is_dir(), f'{DATA_ROOT} not found in Google Drive (wrong account, or the folder is missing)'
for folder in ['raw', 'processed', 'reports/tables']:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
for tmp in DATA_ROOT.rglob('*.tmp'):   # half-written files from an interrupted session
    tmp.unlink()
print('code commit:', CODE_COMMIT, '| data root:', DATA_ROOT)

In [ ]:
import pandas as pd
from foodwatch import load, panel, features, checks
items = SETTINGS['items']; ids = [i['id'] for i in items]; d = SETTINGS['drivers']
weather_items = [i['id'] for i in items if i['weather']]
fx_items = [i['id'] for i in items if i['fx']]
P = panel.build(load.prices(), ids, SETTINGS['gap_2022'])
D, avail = features.add_drivers(P, load.ramadan_eid(), load.weather(), load.fx(), weather_items=weather_items, fx_items=fx_items,
                                lag_days=d['weather_lag_days'], cap_days=d['carry_cap_days'])
dc = checks.data_checks(D, avail, len(ids), weather_items, fx_items, d['carry_cap_days'])
assert dc['ok'], dc

In [ ]:
# Regression check: the same 8 items in the earlier run's panel and features (old Drive folder). Skipped if those files are not there.
from pathlib import Path
OLD = Path(SETTINGS['drive']['old_processed'])
if (OLD / 'tcb_v2_rebuilt.parquet').exists() and (OLD / 'features_v2.parquet').exists():
    rc = checks.compare_with_old(D, pd.read_parquet(OLD / 'tcb_v2_rebuilt.parquet'), pd.read_parquet(OLD / 'features_v2.parquet'))
    assert rc['ok'], rc
else:
    rc = {'skipped': 'old panel or features not found in ' + str(OLD)}

In [ ]:
h_data = write_csv(D[features.COLUMNS], data('data'))
summary = checks.item_summary(D, {i['id']: i['group'] for i in items})
write_csv(summary, DATA_ROOT / SETTINGS['files']['tables'] / 'D1_panel_summary.csv')
record = dict(code_commit=CODE_COMMIT, data=dict(file=SETTINGS['files']['data'], sha256=h_data, column_names=features.COLUMNS, **dc),
              regression_vs_old=rc, raw_sha256={k: sha256(raw(k)) for k in SETTINGS['raw_files']})
tmp = data('checks').with_suffix('.json.tmp'); tmp.write_text(json.dumps(record, indent=2, default=str)); tmp.replace(data('checks'))

In [ ]:
print('=== 02 data processing summary ===')
print('code commit:', CODE_COMMIT)
print('data:', SETTINGS['files']['data'], '| rows', dc['rows'], '| columns', dc['columns'], '| sha256', h_data[:16], '| checks', 'PASS' if dc['ok'] else 'FAIL')
print({k: v for k, v in dc.items() if k not in ('rows', 'columns', 'ok')})
print('regression vs earlier run:', 'PASS' if rc.get('ok') else rc)
if rc.get('ok'):
    print({k: v for k, v in rc.items() if k.startswith('fx_')})
print(summary.to_string(index=False))